<a href="https://colab.research.google.com/github/Ajaykumar-02/NLP/blob/main/Project(NLP).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [314]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [315]:
df = pd.read_csv('train.txt',sep=';',header=None,names=['text','emotion'])

In [316]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [317]:
df.isnull().sum()

,0
text,0
emotion,0


Map(0,1..) emotions

In [318]:
unique_emotions = df['emotion'].unique()
emotion_numbers = {}
i = 0
for emo in unique_emotions:
  emotion_numbers[emo] = i
  i +=1

df['emotion'] = df['emotion'].map(emotion_numbers)

In [319]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


Step 1 : Text into lowercase(abc..)

In [320]:
df['text'] = df['text'].apply(lambda x: x.lower())

Step 2 : Remove Punctuation(@,&..) from text

In [321]:
import string

def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))


In [322]:
df['text'] = df['text'].apply(remove_punc)

Step 3 : Remove Numbers from text

In [323]:

def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_numbers)

Step 4 : Remove Emojis

In [324]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)

Step 5 : Remove Stopwords (is,the..) very imp

In [325]:
import nltk

In [326]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [327]:
stop_words =set(stopwords.words('english'))


In [328]:
len(stop_words)

198

In [329]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [330]:
def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)

  return ' '.join(cleaned)

In [331]:
df['text'] = df['text'].apply(remove)


In [332]:


df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [333]:
from sklearn.model_selection import train_test_split

In [334]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)

In [335]:
X_train

,text
676,refers course though cant help feeling somehow...
12113,im starting feel im suffering fatigue
7077,feel like probably would liked book little bit...
13005,really feel awkward
12123,im feeling little grumpy today lame weather te...
...,...
13418,love leave reader feeling confused slightly de...
5390,feel delicate
860,starting feel little stressed
15795,feel stressed tired worn shape neglected


In [336]:
X_test

,text
8756,ive made week feel beaten
4660,feel strategy worthwhile
6095,feel worthless weak say want find
304,feel clever nov
8241,im moved ive feeling kind gloomy
...,...
15578,feel useful pulpit find ironic often question ...
5746,dried bladders ready day im feeling brave
6395,feel thrilled matter days
7624,woke morning text mr c declaring walking work ...


In [337]:
df.shape

(16000, 2)

Model Implimentation : Naive Bayes

In [338]:

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)


nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)


pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))

0.768125


In [339]:
pred_bow

array([0, 5, 0, ..., 5, 5, 0])

Tf-idf

In [342]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf, y_train)

pred_tfidf = nb2_model.predict(X_test_tfidf)
print(accuracy_score(y_test, pred_tfidf))

0.6609375


Model Implimentation : Logistic Regression

In [343]:
from sklearn.linear_model import LogisticRegression

In [345]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_tfidf, y_train)



LogisticRegression(max_iter=1000)

In [346]:
pred_log = log_model.predict(X_test_tfidf)
print(accuracy_score(y_test, pred_log))

0.8628125


Model Implimentation : SVM

In [347]:
from sklearn.svm import SVC

In [348]:
svm_model = SVC()
svm_model.fit(X_train_tfidf, y_train)

SVC()

In [350]:
pred_svm = svm_model.predict(X_test_tfidf)
print(accuracy_score(y_test, pred_svm))

0.85125


In [352]:
import pickle

pickle.dump(log_model, open("emotion_model.pkl", "wb"))
pickle.dump(tfidf_vectorizer, open("tfidf.pkl", "wb"))
pickle.dump(emotion_numbers, open("emotion_map.pkl", "wb"))